# Kiểm tra dữ liệu bị mờ/lỗi với MTCNN
Notebook này sẽ dùng MTCNN để quét lại các ảnh. Những ảnh nào MTCNN không thể tìm thấy khuôn mặt sẽ được copy vào thư mục `check/` để bạn xem lại và gán nhãn xóa nếu cần.

In [1]:
import os
import shutil
import math
import matplotlib.pyplot as plt
from PIL import Image
from facenet_pytorch import MTCNN
from pathlib import Path

# Khởi tạo MTCNN (nếu có GPU thì sửa device='cuda')
mtcnn = MTCNN(keep_all=False, device='cpu')

dataset_dir = Path("../datasets/LCC_FASD")
check_dir = Path("../check")

# Xóa thư mục check cũ nếu có và tạo lại
if check_dir.exists():
    shutil.rmtree(check_dir)
check_dir.mkdir(parents=True, exist_ok=True)

/home/dainguyen1506/Documents/Project/fas/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
all_images = []
for split in ['train', 'val', 'test']:
    for label in ['real', 'spoof']:
        cls_dir = dataset_dir / split / label
        if cls_dir.exists():
            all_images.extend(list(cls_dir.glob("*.png")))

print(f"Tổng số ảnh sẽ quét: {len(all_images)}")

failed_images = []

for i, img_path in enumerate(all_images):
    try:
        img = Image.open(img_path).convert('RGB')
        boxes, _ = mtcnn.detect(img)
        if boxes is None:
            failed_images.append(img_path)
            shutil.copy2(img_path, check_dir / img_path.name)
    except Exception as e:
        print(f"Lỗi ở file {img_path.name}: {e}")
    
    if (i + 1) % 1000 == 0:
        print(f"Đã xử lý {i + 1} / {len(all_images)}")

print(f"\nQuét xong! Có {len(failed_images)} ảnh không nhận diện được.")

Tổng số ảnh sẽ quét: 18827
Đã xử lý 1000 / 18827
Đã xử lý 2000 / 18827
Đã xử lý 3000 / 18827
Đã xử lý 4000 / 18827
Đã xử lý 5000 / 18827
Đã xử lý 6000 / 18827
Đã xử lý 7000 / 18827
Đã xử lý 8000 / 18827


## Kiểm tra thủ công

Dưới đây là kiểm tra thủ công nhằm lọc ra những bức ảnh không có mặt người nhằm xóa bớt chúng ra khỏi dữ liệu

In [ ]:
check_images = sorted([f for f in os.listdir(check_dir) if f.endswith('.png')])

if len(check_images) == 0:
    print("Không có ảnh nào bị lỗi!")
else:
    num_cols = 4
    num_rows = math.ceil(len(check_images) / num_cols)
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(16, 4 * num_rows))
    
    if type(axes) is not list and type(axes) is not tuple and hasattr(axes, 'flatten'):
        axes = axes.flatten()
    elif type(axes) is not list and type(axes) is not tuple:
        axes = [axes]
        
    for i, img_name in enumerate(check_images):
        img_path = check_dir / img_name
        img = Image.open(img_path)
        axes[i].imshow(img)
        axes[i].set_title(img_name, fontsize=10)
        axes[i].axis('off')
        
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
        
    plt.tight_layout()
    plt.show()

## Xóa ảnh thủ công
Nhập tên các bức ảnh bạn nhìn thấy ở trên mà bạn muốn xóa vào list dưới đây và chạy cell cuối cùng.

In [ ]:
images_to_delete = [
    # Copy và paste tên file vào đây, ví dụ:
    # "test_spoof_spoof_1234.png",
    # "train_real_5678.png"
]

In [ ]:
deleted_count = 0

for img_name in images_to_delete:
    found = False
    for split in ['train', 'val', 'test']:
        for label in ['real', 'spoof']:
            original_path = dataset_dir / split / label / img_name
            if original_path.exists():
                os.remove(original_path)
                print(f"Đã xóa: {original_path}")
                found = True
                deleted_count += 1
                break
        if found:
            break
            
    if not found:
        print(f"Không tìm thấy file trong dataset: {img_name}")
        
print(f"\nĐã xóa thành công {deleted_count} ảnh.")